# Data Preparation for Predictive Maintenance Project

This notebook prepares the datasets for the predictive maintenance project. It includes:

- Splitting the synthetic predictive maintenance data into training and testing sets.
- Adding a timestamp and preprocessing the equipment anomaly data.

**Note:** Adjust file paths as necessary if you move the datasets or output files.

In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Paths to original datasets
predictive_path = 'data/predictive_maintenance.csv'
anomaly_path = 'data/equipment_anomaly_data.csv'

# Load datasets
predictive_df = pd.read_csv(predictive_path)
anomaly_df = pd.read_csv(anomaly_path)


In [2]:

# Split predictive maintenance dataset into 80% train and 20% test
train_df, test_df = train_test_split(
    predictive_df,
    test_size=0.20,
    stratify=predictive_df['Target'],
    random_state=42
)

# Save the split datasets
train_df.to_csv('train_predictive_maintenance.csv', index=False)
test_df.to_csv('test_predictive_maintenance.csv', index=False)

print(f'Training set shape: {train_df.shape}')
print(f'Testing set shape: {test_df.shape}')


Training set shape: (8000, 10)
Testing set shape: (2000, 10)


In [3]:

# Add a timestamp column with hourly increments starting from 2024-01-01 00:00:00
start_time = pd.Timestamp('2024-01-01 00:00:00')
anomaly_df['timestamp'] = [start_time + pd.Timedelta(hours=i) for i in range(len(anomaly_df))]

# Rename columns for consistency
rename_map = {
    'temperature': 'temp_celsius',
    'pressure': 'pressure_psi',
    'vibration': 'vibration',
    'humidity': 'humidity_pct',
    'equipment': 'equipment_type',
    'location': 'location',
    'faulty': 'target'
}
anomaly_df = anomaly_df.rename(columns=rename_map)

# Ensure target is integer type
anomaly_df['target'] = anomaly_df['target'].astype(int)

# Save the timestamped (raw) dataset
anomaly_df.to_csv('timestamped_equipment_anomaly_data.csv', index=False)

anomaly_df.head()


,temp_celsius,pressure_psi,vibration,humidity_pct,equipment_type,location,target,timestamp
0,58.180180,25.029278,0.606516,45.694907,Turbine,Atlanta,0,2024-01-01 00:00:00
1,75.740712,22.954018,2.338095,41.867407,Compressor,Chicago,0,2024-01-01 01:00:00
2,71.358594,27.276830,1.389198,58.954409,Turbine,San Francisco,0,2024-01-01 02:00:00
3,71.616985,32.242921,1.770690,40.565138,Pump,Atlanta,0,2024-01-01 03:00:00
4,66.506832,45.197471,0.345398,43.253795,Pump,New York,0,2024-01-01 04:00:00


In [4]:

# One-hot encode categorical variables
categorical_cols = ['equipment_type', 'location']
anomaly_encoded = pd.get_dummies(anomaly_df, columns=categorical_cols, drop_first=False)

# Standardize numeric features
numeric_cols = ['temp_celsius', 'pressure_psi', 'vibration', 'humidity_pct']
scaler = StandardScaler()
anomaly_encoded[numeric_cols] = scaler.fit_transform(anomaly_encoded[numeric_cols])

# Save the preprocessed dataset
anomaly_encoded.to_csv('preprocessed_equipment_anomaly_data.csv', index=False)

anomaly_encoded.head()


,temp_celsius,pressure_psi,vibration,humidity_pct,target,timestamp,equipment_type_Compressor,equipment_type_Pump,equipment_type_Turbine,location_Atlanta,location_Chicago,location_Houston,location_New York,location_San Francisco
0,-0.786610,-1.031582,-1.379925,-0.364984,0,2024-01-01 00:00:00,False,False,True,True,False,False,False,False
1,0.297440,-1.231493,0.996943,-0.688233,0,2024-01-01 01:00:00,True,False,False,False,True,False,False,False
2,0.026922,-0.815074,-0.305569,0.754840,0,2024-01-01 02:00:00,False,False,True,False,False,False,False,True
3,0.042873,-0.336688,0.218089,-0.798215,0,2024-01-01 03:00:00,False,True,False,True,False,False,False,False
4,-0.272588,0.911232,-1.738352,-0.571147,0,2024-01-01 04:00:00,False,True,False,False,False,False,True,False
